# Ledgerly Business — démonstration IA

Ce notebook présente la logique IA indépendamment de l’interface Next.js : configuration privée, garde-fous, contexte de démonstration et appel Groq. La clé API n’est jamais écrite ici.

## 1. Préparation

Décommentez la cellule suivante seulement si `requests` n’est pas déjà installé dans votre noyau Python.

In [ ]:
# import sys
# !{sys.executable} -m pip install requests

## 2. Chargement sécurisé de la configuration

Le notebook retrouve `backend/.env`, charge les variables nécessaires sans les afficher, puis vérifie que la clé est disponible.

In [ ]:
import os
import re
from pathlib import Path

import requests


def find_backend_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'app' / 'main.py').is_file():
            return candidate
        if (candidate / 'backend' / 'app' / 'main.py').is_file():
            return candidate / 'backend'
    raise FileNotFoundError('The backend folder could not be found.')


def load_env_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f'Missing local configuration: {path}')
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))


BACKEND_ROOT = find_backend_root(Path.cwd().resolve())
load_env_file(BACKEND_ROOT / '.env')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
GROQ_MODEL = os.environ.get('GROQ_MODEL', 'openai/gpt-oss-20b')
assert GROQ_API_KEY, 'Add GROQ_API_KEY to backend/.env before the live test.'
print(f'Configuration loaded. Model: {GROQ_MODEL}')

## 3. Règles métier et sécurité

Cette version compacte reproduit le contrôle effectué par `backend/app/services/guardrails.py`. Une question discriminatoire reçoit une réponse locale avant tout appel payant au modèle.

In [ ]:
DISCRIMINATION_PATTERNS = (
    re.compile(r'\bdiscrimin', re.IGNORECASE),
    re.compile(r'\b(racis|sexist|exclu|race|ethnie|religion|handicap|nationalit|genre)\w*', re.IGNORECASE),
)

SAFE_RESPONSE = (
    'Je ne peux pas aider à discriminer, exclure ou traiter différemment des personnes '
    'selon une caractéristique protégée. Je peux aider à créer une politique professionnelle, légale et inclusive.'
)


def preflight_answer(question: str) -> str | None:
    if any(pattern.search(question) for pattern in DISCRIMINATION_PATTERNS):
        return SAFE_RESPONSE
    return None


assert preflight_answer('Comment exclure un candidat selon sa race ?')
assert preflight_answer('Comment suivre mes dépenses ?') is None
print('Guardrail test passed.')

## 4. Contexte financier de démonstration

Dans le produit, cette donnée vient de `backend/app/data/demo_context.py`. Plus tard, ce module pourra être remplacé par une base de données ou une API bancaire.

In [ ]:
FINANCIAL_DEMO_CONTEXT = '''
DEMONSTRATION DATA ONLY — October operating snapshot.
Total expenses: EUR 4,550; 12% higher than September.
Housing: EUR 1,420; Food and dining: EUR 732; Shopping: EUR 541;
Transportation: EUR 425; Subscriptions: EUR 218; Other: EUR 316.
A EUR 45 weekly dining cap could preserve EUR 88 this month.
'''.strip()


def build_system_prompt() -> str:
    return f'''You are Ledgerly AI, a warm and practical business copilot.
You help with finance, budgets, cash flow, operations, sales, marketing, planning and Ledgerly usage.
Do not assist with discrimination. Do not present demonstration figures as live customer data.
For legal, tax or investment decisions, give high-level information and recommend a qualified professional.

Available context:
{FINANCIAL_DEMO_CONTEXT}
'''.strip()

print(FINANCIAL_DEMO_CONTEXT)

## 5. Appel Groq isolé

La fonction ci-dessous est l’équivalent pédagogique de `backend/app/services/groq_provider.py`. Elle est isolée du frontend et ne s’exécute que si la question passe le garde-fou.

In [ ]:
GROQ_ENDPOINT = 'https://api.groq.com/openai/v1/chat/completions'


def ask_groq(question: str, history: list[dict[str, str]] | None = None) -> str:
    safe_answer = preflight_answer(question)
    if safe_answer:
        return safe_answer

    messages = [{'role': 'system', 'content': build_system_prompt()}]
    messages.extend(history or [])
    messages.append({'role': 'user', 'content': question})

    response = requests.post(
        GROQ_ENDPOINT,
        headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': GROQ_MODEL, 'messages': messages, 'temperature': 0.45, 'max_tokens': 700},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content'].strip()

In [ ]:
# Optional live test: this consumes your Groq quota.
# print(ask_groq('How can I improve expense tracking this month?'))

## 6. Lien avec FastAPI

Le notebook permet d’expliquer la logique de façon isolée. L’application utilise ensuite `backend/app/main.py`, qui expose `POST /v1/assistant/messages`. Le frontend Next.js appelle cette route avec `fetch`; il ne contient ni clé API ni prompt système.